# Path-only models on Real-IM (Kaggle GPU): train / evaluate

Models from `models/path_models.py` (plan: `CLAUDE/pose_conditioned_path.md`):
`real_PathUNet` (B2 baseline), `real_PathUNetPose` (M1, PathNet + pose history).

Before running: **Settings -> Accelerator -> GPU T4 x2**, **Settings -> Internet -> On**, and **Add Input**:
* the Real-IM dataset (the 5 `.npz` files)
* the released PathNet weights (`pathnet_3600.pt`) for `INIT="released"` or B1 evaluation
* to resume: the previous version's output of this notebook (it contains `trainstate_last.pt`)

Start runs with **Save Version -> Save & Run All (Commit)**. Everything in `/kaggle/working` becomes the version's output.

In [ ]:
# ===== Parameters: the only cell you normally edit =====
REPO_URL = "https://github.com/ThanasisGeo/front_following_iwalk.git"
BRANCH = "main"
USE_GITHUB_TOKEN = False      # True for a private repo: add a Kaggle Secret named GITHUB_TOKEN

CFG = "real_PathUNetPose"     # real_PathUNet (B2) | real_PathUNetPose (M1) | real_PathUNetPoseMapOnly (M1, no past-path heat-maps)
EXP = "m1_rel_s0"             # run name -> results/<CFG>_<EXP>/  (suggested: <model>_<init>_s<seed>[_f<testseq>])
SEED = 0
INIT = "released"             # "released": warm start from pathnet_3600.pt | "none": train from scratch
LR = 1e-5                     # 1e-5 for warm starts; 0 = cfg lr (1e-4, from scratch)
EPOCHS = 600                  # TOTAL epochs; a resumed run continues up to this number
BATCH_SIZE = 32
TEST_SEQ = None               # None = cfg test list (me_f2_seq1_raw). Another sequence = a fold (INIT="none" only)
SELECT = "val_ade"            # validation metric that picks pathnet_best.pt: val_ade | val_fde | val_turn_fde
EXTRA_SET = []                # cfg overrides, e.g. ["model_specs.n_feats_pose_path=64"]
MODE = "smoke"                # smoke: 2 epochs to measure speed | train | eval_only (evaluate the released weights)
TIME_LIMIT_H = 11             # stop cleanly (and save) before Kaggle's 12 h session limit
KEEP_ONLY_BEST = False        # True: delete periodic pathnet_<N>.pt at the end (keeps best + trainstate_last)

In [ ]:
# ===== 1. Hardware check =====
import sys, subprocess, torch
print(sys.version)
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    arch = f"sm_{cap[0]}{cap[1]}"
    print(torch.cuda.get_device_name(0), arch, "| torch built for:", torch.cuda.get_arch_list())
    assert arch in torch.cuda.get_arch_list(), (
        f"This torch build has no kernels for {arch} (the P100 is sm_60). Switch the accelerator to GPU T4 x2.")
else:
    print("WARNING: no GPU. Fine for MODE='eval_only'; training the UNet models on CPU is very slow.")

In [ ]:
# ===== 2. Code + the one missing package =====
import os, glob, shutil
REPO = "/kaggle/working/iros2024_poseForecasting"
url = REPO_URL
if USE_GITHUB_TOKEN:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)  # keep the token out of the output
commit = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open3d"], check=True)  # imported by the data loader

In [ ]:
# ===== 3. Inputs: data, released weights, resume state =====
def find_one(pattern, what):
    # Search all inputs by file name; skip earlier runs' outputs (results/), which can hold same-named checkpoints.
    hits = [h for h in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True) if "/results/" not in h]
    assert len(hits) == 1, f"{what}: expected exactly one '{pattern}' under /kaggle/input, found {hits}"
    return hits[0]

# Data -> data/Real-IM-Dataset-processed/ (where the cfg expects it)
data_src = os.path.dirname(find_one("kl_f2_seq1_raw.npz", "Real-IM dataset (Add Input -> your dataset)"))
data_dst = f"{REPO}/data/Real-IM-Dataset-processed"
os.makedirs(os.path.dirname(data_dst), exist_ok=True)
if os.path.islink(data_dst):
    os.unlink(data_dst)
os.symlink(data_src, data_dst)
print("data:", data_src, sorted(os.listdir(data_src)))

RELEASED = None
if INIT == "released" or MODE == "eval_only":
    RELEASED = find_one("pathnet_3600.pt", "released PathNet weights (Add Input -> dataset with pathnet_3600.pt)")
    print("released weights:", RELEASED)
if INIT == "released" and TEST_SEQ:
    raise ValueError("The released PathNet was trained on every sequence except me_f2_seq1_raw, so a warm start "
                     "leaks the test data of any other fold. Use INIT='none' for folds.")

# Resume: the previous version's output (attached as input) contains this run's trainstate_last.pt
RUN = f"{CFG}_{EXP}"
run_dir = f"{REPO}/results/{RUN}"
RESUME = None
prev = glob.glob(f"/kaggle/input/**/results/{RUN}/models/trainstate_last.pt", recursive=True)
if MODE == "train" and prev:
    assert len(prev) == 1, f"several previous runs found: {prev}"
    prev_dir = os.path.dirname(os.path.dirname(prev[0]))
    os.makedirs(f"{run_dir}/models", exist_ok=True)
    for rel in ["models/trainstate_last.pt", "models/pathnet_best.pt", "models/best.json", "train_metrics.csv"]:
        if os.path.exists(f"{prev_dir}/{rel}"):
            shutil.copy(f"{prev_dir}/{rel}", f"{run_dir}/{rel}")
    RESUME = f"{run_dir}/models/trainstate_last.pt"
    print("resuming from", prev_dir)

In [ ]:
# ===== 4. Train =====
import csv
env = "TQDM_DISABLE=1"          # no progress bars in the saved log
if MODE in ("smoke", "train"):
    run_exp = EXP + "_smoke" if MODE == "smoke" else EXP
    log_dir = f"{REPO}/results/{CFG}_{run_exp}"
    os.makedirs(log_dir, exist_ok=True)
    args = [f"--cfg {CFG}", f"--exp {run_exp}", f"--seed {SEED}", f"--epochs {2 if MODE == 'smoke' else EPOCHS}",
            f"--batch_size {BATCH_SIZE}", f"--select {SELECT}", "--num_workers 4"]
    if LR:
        args.append(f"--lr {LR}")
    if TEST_SEQ:
        args.append(f"--test_seq {TEST_SEQ}")
    if RESUME:
        args.append(f"--resume {RESUME}")
    elif INIT == "released":
        args.append(f"--init_released {RELEASED}")
    if EXTRA_SET:
        args.append("--set " + " ".join(EXTRA_SET))
    cmd = (f"set -o pipefail; cd {REPO} && {env} timeout {TIME_LIMIT_H}h {sys.executable} -u train_path.py "
           + " ".join(args) + f" 2>&1 | tee -a {log_dir}/train_log.txt")
    print(cmd)
    rc = subprocess.run(["bash", "-c", cmd]).returncode
    if rc == 124:
        print("Stopped by the time limit after saving. Attach this version's output and run again to resume.")
    elif rc != 0:
        raise RuntimeError(f"train_path.py failed (exit {rc}), see the log above")
    if MODE == "smoke":
        secs = [float(r["seconds"]) for r in csv.DictReader(open(f"{log_dir}/train_metrics.csv"))]
        per_epoch = secs[-1]   # the first epoch includes start-up costs
        print(f"\n~{per_epoch:.1f} s/epoch -> {EPOCHS} epochs = {EPOCHS * per_epoch / 3600:.1f} h; "
              f"~{int(TIME_LIMIT_H * 3600 / per_epoch)} epochs fit in one {TIME_LIMIT_H} h session")

In [ ]:
# ===== 5. Evaluate on the test sequence (turn subsets, constant-velocity baseline) =====
def eval_cmd(ckpt, exp):
    extra = f" --test_seq {TEST_SEQ}" if TEST_SEQ else ""
    return (f"set -o pipefail; cd {REPO} && {env} {sys.executable} -u eval_path.py --cfg {CFG} --exp {exp} "
            f"--ckpt {ckpt}{extra} 2>&1 | grep -v '^read\\|^generating\\|^seq length' "
            f"| tee -a {REPO}/results/{CFG}_{exp}/eval_log.txt")

if MODE == "eval_only":
    os.makedirs(run_dir, exist_ok=True)
    subprocess.run(["bash", "-c", eval_cmd(RELEASED, EXP)], check=True)
elif MODE == "train":
    for ckpt in ("best", "last"):
        subprocess.run(["bash", "-c", eval_cmd(ckpt, EXP)], check=True)

In [ ]:
# ===== 6. Clean the output =====
if os.path.islink(data_dst):
    os.unlink(data_dst)  # don't save a link to /kaggle/input
out_dir = f"{REPO}/results/{CFG}_{EXP}_smoke" if MODE == "smoke" else run_dir  # the folder this run wrote to
os.makedirs(out_dir, exist_ok=True)
if KEEP_ONLY_BEST and MODE == "train":
    for p in glob.glob(f"{out_dir}/models/pathnet_[0-9]*.pt"):
        os.remove(p)
with open(f"{out_dir}/code_commit.txt", "a") as f:
    f.write(f"{commit} mode={MODE} init={INIT} test_seq={TEST_SEQ}\n")
subprocess.run(["bash", "-c", f"du -sh /kaggle/working; ls -la {out_dir} {out_dir}/models 2>/dev/null"])